# 04g — The Ratchet R-Test (clean vs dirty interventions)

> Attacks the load-bearing DPI-for-interventions lemma (R ≤ 0) at toy rank.
> See `../canon/00-foundations/04-break-even-theorem.md` §proof program.

## CONTRACT (frozen 2026-07-18)

- **PREDICTION**: a CLEAN scramble (belief-only, marginal-preserving) yields a stable
  work-per-bit ratio; a DIRTY scramble (intervention also perturbs the WORLD) inflates
  apparent work-per-bit — the R > 0 signature the lemma must exclude.
- **BASELINE**: clean-scramble ratio.
- **DATA**: seeded ring world, 8 seeds × p ∈ {0.90, 0.95, 0.98}.
- **PASS**: dirty excess > 2σ (demonstrates the lemma's boundary: interventions must touch
  ONLY the information channel or Π_A over-counts).
- **FALSIFIER**: no dirty excess → the protocol's validity condition is vacuous here.


In [ ]:
import numpy as np

C_SENSE, C_MEMORY, C_MOVE = 0.010, 0.020, 0.005

def run_ring(p, use_memory, scramble=None, steps=4000, seed=65537):
    """scramble: None | 'clean' (belief only, marginal-preserving) | 'dirty' (also kicks the WORLD).
    Returns (gathered, model_cost, info_bits) where info_bits = empirical I(believed;direction) proxy
    = per-step accuracy-based mutual information of the 1-bit belief channel."""
    rng = np.random.default_rng(seed); srng = np.random.default_rng(seed + 1)
    n = 64; peak, direction = 0, 1
    pos, gathered, model_cost = 0, 0.0, 0.0
    believed, prev_peak = 1, 0
    agree = 0
    def resource(p_, peak_):
        d = min(abs(p_ - peak_), n - abs(p_ - peak_))
        return max(0.0, 1.0 - d / 8.0)
    for _ in range(steps):
        gathered -= C_SENSE
        if use_memory:
            model_cost += C_MEMORY; gathered -= C_MEMORY
            delta = (peak - prev_peak + n//2) % n - n//2
            if delta: believed = 1 if delta > 0 else -1
            if scramble in ('clean', 'dirty'):
                believed = srng.choice([-1, 1])
                if scramble == 'dirty':
                    # NON-INFORMATIONAL side effect: the intervention also kicks the world
                    peak = (peak + srng.choice([-1, 1])) % n
            target = (peak + believed) % n
        else:
            target = peak
        prev_peak = peak
        offset = (target - pos + n//2) % n - n//2
        if offset:
            gathered -= C_MOVE
            pos = (pos + (1 if offset > 0 else -1)) % n
        agree += (believed == direction)
        if rng.random() > p: direction = -direction
        peak = (peak + direction) % n
        gathered += resource(pos, peak)
    # 1-bit channel: I = 1 - H(acc) (bits/step), acc = P(believed == direction)
    acc = agree / steps
    eps = 1e-12
    H = -(acc*np.log2(acc+eps) + (1-acc)*np.log2(1-acc+eps))
    return gathered, model_cost, (1.0 - H) * steps

SEEDS = [65537 + 1000*k for k in range(8)]
print(f"{'p':>6} {'dW_clean':>9} {'dI_clean':>9} {'r_clean':>8} {'dW_dirty':>9} {'dI_dirty':>9} {'r_dirty':>8}")
ratios_clean, ratios_dirty = [], []
for p in (0.90, 0.95, 0.98):
    for s in SEEDS:
        wi, cm, Ii = run_ring(p, True, None, seed=s)
        wc, _, Ic = run_ring(p, True, 'clean', seed=s)
        wd, _, Id = run_ring(p, True, 'dirty', seed=s)
        dWc, dIc = wi - wc, Ii - Ic
        dWd, dId = wi - wd, Ii - Id
        if dIc > 1: ratios_clean.append(dWc / dIc)
        if dId > 1: ratios_dirty.append(dWd / dId)
    print(f"{p:>6} {np.mean([wi-wc]):>9.1f} {dIc:>9.1f} {dWc/dIc:>8.3f} {wi-wd:>9.1f} {dId:>9.1f} {dWd/dId:>8.3f}")
rc, rd = np.array(ratios_clean), np.array(ratios_dirty)
print(f"\nwork-per-bit ratio, CLEAN scramble: {rc.mean():.3f} +- {rc.std(ddof=1):.3f}  (n={len(rc)})")
print(f"work-per-bit ratio, DIRTY scramble: {rd.mean():.3f} +- {rd.std(ddof=1):.3f}  (n={len(rd)})")
excess = rd.mean() - rc.mean()
sig = excess / np.hypot(rc.std(ddof=1)/np.sqrt(len(rc)), rd.std(ddof=1)/np.sqrt(len(rd)))
print(f"\nDIRTY excess (the R>0 signature): {excess:.3f} work-units/bit  ({sig:.1f} sigma)")
print("INTERPRETATION:", "dirty intervention over-counts information value -> ablation protocol MUST touch only the information channel (lemma boundary demonstrated)" if sig > 2 else "no significant excess — dirty kick too weak, strengthen the perturbation")


## Result (run 2026-07-18)

```
work-per-bit, CLEAN: 0.102 ± 0.021   (stable across p — itself an invariant candidate)
work-per-bit, DIRTY: 0.344 ± 0.101
DIRTY excess: +0.242 work-units/bit  (11.6σ)  →  PASS
```

A dirty intervention inflates apparent information value 3.4×. At toy rank this
demonstrates exactly the failure mode R > 0: work changes not mediated by information
loss contaminate the ablation delta. Protocol law: **the intervention must be a
do-operation on the information channel alone.** The clean ratio's stability (~0.10
across p) is a candidate for the cost-rescaled invariant (see 04f control).


## ⚠️ Round-3 verification note (2026-07-18)

Reframed per panel verdict: the dirty-scramble result is a **negative control / instrument validation**
(near-trivial as a lemma test — it demonstrates the detector, not the theorem). The load-bearing
observation is the **clean** work-per-bit stability (0.102 ± 0.021 across p) — weak positive evidence
for R ≤ 0. The non-trivial counterexample to build next: memory-state-dependent potential /
actuator back-action confined provably to the information channel (world transition operator
byte-identical) — either outcome is a result: R ≤ 0 confirms the lemma for I_use; R > 0 promotes
the signed functional J from conjecture to necessity.


In [ ]:
"""04g v2 — CLEAN-CHANNEL R-TEST (Boss #4).
Tests the DPI-for-interventions lemma  R <= 0  where
    R = (W_intact - W_scr) - kT * dI_use.

Design (panel R3 spec): the memory register is PHYSICALLY coupled to the actuator —
holding a particular bit pattern sets an energy BARRIER, independent of what that
pattern predicts. The world's transition operator is byte-identical under intact vs
scrambled (verified explicitly below), so the intervention touches ONLY the
information channel; yet work changes through a static energetic path that pairwise
MI between the decision and the work coordinate does not register.

If R > 0 robustly here: the lemma is FALSE for I_use, and the signed functional J
(which carries the back-action term) is NECESSARY, not optional.
"""
import numpy as np

KT = 1.0

def _mi(a, b, nb=8):
    a = np.asarray(a, float); b = np.asarray(b, float)
    def disc(v):
        u = np.unique(v)
        if len(u) <= 6: return np.searchsorted(u, v)
        return np.searchsorted(np.quantile(v, np.linspace(0,1,nb+1)[1:-1]), v)
    ia, ib = disc(a), disc(b)
    def raw(x, y):
        H = np.zeros((x.max()+1, y.max()+1)); np.add.at(H, (x, y), 1.0)
        P = H/H.sum(); Px = P.sum(1, keepdims=True); Py = P.sum(0, keepdims=True)
        nz = P > 0
        return float(np.sum(P[nz]*np.log2(P[nz]/(Px@Py)[nz])))
    rng = np.random.default_rng(999)
    null = np.mean([raw(ia, rng.permutation(ib)) for _ in range(8)])
    return max(0.0, raw(ia, ib) - null)

def run(mode, steps=20000, seed=0, barrier_strength=0.0, p=0.95):
    """mode: 'intact' | 'scrambled'.
    barrier_strength = 0 -> pure information channel (control condition)
    barrier_strength > 0 -> memory register ALSO sets an energy barrier (back-action)
    World stream is drawn from its OWN rng, advanced identically in both modes."""
    wrng = np.random.default_rng(seed)          # world stream — identical both modes
    srng = np.random.default_rng(seed + 1)
    n = 64
    peak, direction = 0, 1
    pos = 0
    believed, prev_peak = 1, 0
    W = 0.0
    D, R_inc = [], []
    world_trace = []
    for _ in range(steps):
        delta = (peak - prev_peak + n//2) % n - n//2
        if delta: believed = 1 if delta > 0 else -1
        b = srng.choice([-1, 1]) if mode == 'scrambled' else believed
        # --- the memory register's PHYSICAL side effect: it sets a barrier ---
        # cost depends on the STORED PATTERN itself (b), not on what it predicts
        barrier = barrier_strength * (1.0 if b > 0 else 0.0)
        target = (peak + b) % n
        prev_peak = peak
        off = (target - pos + n//2) % n - n//2
        step = 1 if off > 0 else (-1 if off < 0 else 0)
        if step:
            W -= 0.005 + barrier            # actuation pays the barrier
            pos = (pos + step) % n
        # world advances from its own stream: byte-identical across modes
        u = wrng.random()
        world_trace.append(u)
        if u > p: direction = -direction
        peak = (peak + direction) % n
        d = min(abs(pos-peak), n-abs(pos-peak))
        r = max(0.0, 1.0 - d/8.0)
        W += r
        D.append(float(step)); R_inc.append(r)
    return W, _mi(D, R_inc)*steps, np.array(world_trace)

SEEDS = [65537 + 1000*k for k in range(8)]
print("CLEAN-CHANNEL R-TEST — R = (W_intact - W_scr) - kT*dI_use")
print("="*76)
for bs in (0.000, 0.020, 0.060):
    Rs = []
    for s in SEEDS:
        wi, Ii, t1 = run('intact', seed=s, barrier_strength=bs)
        ws, Is, t2 = run('scrambled', seed=s, barrier_strength=bs)
        assert np.array_equal(t1, t2), "world stream differed — intervention NOT clean!"
        Rs.append((wi - ws) - KT*(Ii - Is))
    Rs = np.array(Rs)
    sem = Rs.std(ddof=1)/np.sqrt(len(Rs))
    verdict = "R > 0  (LEMMA VIOLATED)" if Rs.mean() > 2*sem else ("R <= 0 (consistent with lemma)" if Rs.mean() < -2*sem else "R ~ 0 (indeterminate)")
    print(f"barrier={bs:.3f}: R = {Rs.mean():+10.2f} +- {sem:6.2f} (SEM)   {Rs.mean()/sem:+7.2f} sigma   {verdict}")
print("="*76)
print("World-stream identity asserted every run: the intervention provably touches")
print("ONLY the memory register. Any R != 0 is therefore off-shell work, not a leak.")

# ---------------- DIFFERENTIAL TEST (dimensionally honest) ----------------
# Absolute R needs a work<->bit calibration that does not exist in sim units.
# But dR/d(barrier) does NOT: if ALL work changes flow through the information
# channel, then adding a purely energetic back-action must leave R unchanged.
# Paired across identical seeds, so the calibration constant cancels.
print()
print("DIFFERENTIAL TEST: does a purely energetic back-action move R?")
print("="*76)
base = {}
for bs in (0.000, 0.020, 0.060, 0.120):
    vals = []
    for s in SEEDS:
        wi, Ii, t1 = run('intact', seed=s, barrier_strength=bs)
        ws, Is, t2 = run('scrambled', seed=s, barrier_strength=bs)
        assert np.array_equal(t1, t2)
        vals.append(((wi-ws) - KT*(Ii-Is), wi-ws, Ii-Is))
    base[bs] = np.array(vals)
b0 = base[0.000]
for bs in (0.020, 0.060, 0.120):
    d_R  = base[bs][:,0] - b0[:,0]      # paired per seed
    d_W  = base[bs][:,1] - b0[:,1]
    d_I  = base[bs][:,2] - b0[:,2]
    semR = d_R.std(ddof=1)/np.sqrt(len(d_R))
    semI = d_I.std(ddof=1)/np.sqrt(len(d_I)) if d_I.std(ddof=1) > 0 else 0.0
    print(f"barrier {bs:.3f} vs 0: dR = {d_R.mean():+8.2f} +- {semR:5.2f} ({d_R.mean()/semR if semR else float('inf'):+6.2f}s) | "
          f"dW = {d_W.mean():+8.2f} | dI_use = {d_I.mean():+8.2f}" + (f" +- {semI:.2f}" if semI else " (identical)"))
print("="*76)
print("If dI_use ~ 0 while dR != 0, the barrier moved work through a NON-informational")
print("path -> R>0-style violation in the differential sense -> the signed functional J")
print("is NECESSARY (I_use alone cannot account for the work change).")


## v2 — THE CLEAN-CHANNEL TEST (Boss #4) — run 2026-07-18

### Design (panel R3 spec, implemented)

The memory register is physically coupled to the actuator: holding a particular bit
pattern raises an energy **barrier**, independent of what that pattern predicts. The
world's random stream is drawn from its own generator and **asserted byte-identical**
between intact and scrambled runs on every run — so the intervention provably touches
nothing but the memory register.

### Absolute test: inconclusive, and instructively so

```
barrier 0.000 -> R = -12204 +- 41      barrier 0.060 -> R = -12217 +- 40
```

R is dominated by the information term at every barrier. This is a **units artifact**:
R = dW - kT*dI_use compares simulation work units against bits x steps, so it needs a
work<->bit calibration that does not exist in these units. *The same disease as eta\*.*

### Differential test: dimensionally honest, and decisive

If **all** work changes flow through the information channel, adding a purely energetic
back-action must leave R unchanged. Paired across identical seeds, the calibration
constant cancels:

| barrier | dR vs 0 | dW | **dI_use** |
|---|---|---|---|
| 0.020 | -4.36 +- 1.49 (-2.93 sigma) | -4.36 | **+0.00 (identical)** |
| 0.060 | -13.07 +- 4.46 (-2.93 sigma) | -13.07 | **+0.00 (identical)** |
| 0.120 | -26.14 +- 8.91 (-2.93 sigma) | -26.14 | **+0.00 (identical)** |

### VERDICT

**The ablation work difference is tunable to arbitrary size by a purely energetic memory
back-action that carries exactly zero information.** dW scales linearly with the barrier;
dI_use registers literally nothing.

The lemma's *sign* (R <= 0) is not violated here - R stays negative. What fails is the
lemma's **content**: the identity "ablated work measures the information's causal
contribution" cannot hold when dW is movable without moving dI_use at all.

**Consequence for the framework's central instrument:** Pi_A = dW_ablation / C_model does
not measure information-mediated work alone. It measures information-mediated work **plus
memory-state-dependent energetics**, and the two are not separable by any MI-based
quantity. The signed functional **J** - which carries the back-action term explicitly - is
therefore **necessary, not optional**. Conjecture promoted to requirement, by construction
rather than by argument.

**Scope guard:** this is a toy demonstration that the contamination *channel exists* and is
unbounded. It does not show that any previously reported Pi_A measurement was contaminated
- the earlier families have no memory-state-dependent potentials. It shows the instrument
has no defense against one.
